In [1]:
import pandas as pd
import numpy as np

 """
    Buat rekomendasi hybrid untuk user_id.
    1. Cari film yang belum ditonton user
    2. Hitung SVD score
    3. Gabungkan dengan RF score
    4. Sort by hybrid score, return top_n
    """

In [ ]:
df = pd.read_csv('cleaned_data/ratings_val_70.csv')

display(df.head())

,userId,movieId,rating,timestamp,datetime,year,month
0,25062,1176,4.0,789652004,1995-01-09 11:46:44+00:00,1995,1
1,30917,1079,3.0,789652009,1995-01-09 11:46:49+00:00,1995,1
2,30917,21,3.0,789652009,1995-01-09 11:46:49+00:00,1995,1
3,30917,47,5.0,789652009,1995-01-09 11:46:49+00:00,1995,1
4,43720,18,4.0,822873600,1996-01-29 00:00:00+00:00,1996,1


In [2]:
df_master = pd.read_csv('cleaned_data/df_master.csv')
display(df_master.head())
print("Jumlah baris di df_master:", len(df_master))

,movieId,title,genres,year,title_clean,genre_list,Action,Adventure,Animation,Children,...,imdbId,tmdbId,rating_count,rating_mean,rating_std,rating_min,rating_max,tags_list,tags_joined,tag_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,"['Adventure', 'Animation', 'Children', 'Comedy...",0,1,1,1,...,114709,862.0,68997,3.8974,0.9215,0.5,5.0,"['children', 'disney', 'animation', 'children'...",children disney animation children disney disn...,1230
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,"['Adventure', 'Children', 'Fantasy']",0,1,0,1,...,113497,8844.0,28904,3.2758,0.9555,0.5,5.0,"['robin williams', 'fantasy', 'robin williams'...",robin williams fantasy robin williams time tra...,573
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,"['Comedy', 'Romance']",0,0,0,0,...,113228,15602.0,13134,3.1394,1.0123,0.5,5.0,"['comedinha de velhinhos engraãƒâ§ada', 'comed...",comedinha de velhinhos engraãƒâ§ada comedinha ...,23
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,"['Comedy', 'Drama', 'Romance']",0,0,0,0,...,114885,31357.0,2806,2.8453,1.1059,0.5,5.0,"['characters', 'slurs', 'based on novel or boo...",characters slurs based on novel or book chick ...,12
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,['Comedy'],0,0,0,0,...,113041,11862.0,13154,3.0596,0.9991,0.5,5.0,"['fantasy', 'pregnancy', 'remake', 'family', '...",fantasy pregnancy remake family steve martin s...,64


Jumlah baris di df_master: 87585


In [8]:
df_movies = pd.read_csv('cleaned_data/movies_clean.csv')
display(df_movies.head())

,movieId,title,genres,year,title_clean,genre_list,Action,Adventure,Animation,Children,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,"['Adventure', 'Animation', 'Children', 'Comedy...",0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,"['Adventure', 'Children', 'Fantasy']",0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,"['Comedy', 'Romance']",0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,"['Comedy', 'Drama', 'Romance']",0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,['Comedy'],0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df_ratings = pd.read_csv('cleaned_data/ratings.csv')
display(df_ratings.head())

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858



#  Endpoint untuk Halaman Trending/Populer

@app.route('/movies/popular', methods=['GET'])
def get_popular_movies():
    try:
        pipeline = [
            {"$group": {"_id": "$movie_id", "total_ratings": {"$sum": 1}}},
            {"$sort": {"total_ratings": -1}},
            {"$lookup": {
                "from": "movies",
                "localField": "_id",
                "foreignField": "movie_id",
                "as": "movie_detail"
            }},
            {"$unwind": "$movie_detail"},
            {"$match": {"movie_detail.poster_url": {"$ne": None}}},
            {"$limit": 20},
            {"$replaceRoot": {"newRoot": "$movie_detail"}}
        ]
        popular_movies = list(ratings_collection.aggregate(pipeline)) # Sesuaikan nama koleksi ratingmu
        
        for movie in popular_movies:
            if '_id' in movie:
                del movie['_id']

        return jsonify({"status": "ok", "movies": popular_movies}), 200
    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 500